# MIMIC-IV Paper-Style Experiment Runner

This notebook runs the six requested window settings with both `cv10` and `cv5_80_20`, gives a short summary after each run, and keeps a shared summary table for final comparison.

In [18]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

## Import and Reload the Backend Modules

This cell reloads the Python backend so the notebook always uses the latest code.

In [19]:
import importlib
from copy import deepcopy

import pandas as pd
from IPython.display import display

import variable_sets
import window_experiment_common

importlib.reload(variable_sets)
importlib.reload(window_experiment_common)

run_experiment = window_experiment_common.run_experiment
OUT_DIR = Path(window_experiment_common.OUT_DIR)
RUN_OUTPUT_DIR = Path(window_experiment_common.RUN_OUTPUT_DIR)
SUMMARY_OUTPUT_DIR = Path(window_experiment_common.SUMMARY_OUTPUT_DIR)
EXPLAIN_OUTPUT_DIR = Path(window_experiment_common.EXPLAIN_OUTPUT_DIR)

pd.set_option("display.max_columns", 200)

## Define the Shared Summary File

All grid runs append one row per horizon to the same summary CSV.

In [20]:
SUMMARY_TABLE_NAME = "mimiciv_notebook_summary.csv"
SUMMARY_PATH = SUMMARY_OUTPUT_DIR / SUMMARY_TABLE_NAME
SUMMARY_PATH

WindowsPath('d:/ESILV_S2/Intern/mimic_paper_reproduction/mimiciv_test/output/grid_summary/mimiciv_notebook_summary.csv')

## Build the 12 Requested Grid Configurations

This cell generates the six requested window settings and expands them over both split modes, producing 12 configurations in a short loop.

In [21]:
WINDOW_SPECS = [
    {"label": "+6_-6", "cohort_before": 6, "cohort_after": 6},
    {"label": "+3_-6", "cohort_before": 6, "cohort_after": 3},
    {"label": "+0_-6", "cohort_before": 6, "cohort_after": 0},
    {"label": "+8_-8", "cohort_before": 8, "cohort_after": 8},
    {"label": "+3_-9", "cohort_before": 9, "cohort_after": 3},
    {"label": "+0_-9", "cohort_before": 9, "cohort_after": 0},
]

SPLIT_SPECS = [
    {"split_mode": "cv10", "split_tag": "cv10"},
    {"split_mode": "cv5_80_20", "split_tag": "cv5"},
]

BASE_GRID_OPTIONS = {
    "include_static_in_root": False,
    "export_explainability": False,
    "negative_alignment_mode": "paper",
    "save_relational_tables": False,
    "save_curve_png": False,
    "display_plots": False,
    "append_summary": True,
    "summary_table_name": SUMMARY_TABLE_NAME,
}

EXPERIMENT_GRID = []
for window in WINDOW_SPECS:
    for split in SPLIT_SPECS:
        cfg = deepcopy(BASE_GRID_OPTIONS)
        cfg.update(
            {
                "task_name": f"MIMIC-IV grid | {window['label']} | {split['split_tag']}",
                "task_prefix": f"mimiciv_{window['label']}_{split['split_tag']}".replace("+", "p").replace("-", "m"),
                "cohort_before": window["cohort_before"],
                "cohort_after": window["cohort_after"],
                "split_mode": split["split_mode"],
                "window_label": window["label"],
            }
        )
        EXPERIMENT_GRID.append(cfg)

pd.DataFrame(EXPERIMENT_GRID)[["task_prefix", "window_label", "cohort_before", "cohort_after", "split_mode"]]

,task_prefix,window_label,cohort_before,cohort_after,split_mode
0,mimiciv_p6_m6_cv10,+6_-6,6,6,cv10
1,mimiciv_p6_m6_cv5,+6_-6,6,6,cv5_80_20
2,mimiciv_p3_m6_cv10,+3_-6,6,3,cv10
3,mimiciv_p3_m6_cv5,+3_-6,6,3,cv5_80_20
4,mimiciv_p0_m6_cv10,+0_-6,6,0,cv10
5,mimiciv_p0_m6_cv5,+0_-6,6,0,cv5_80_20
6,mimiciv_p8_m8_cv10,+8_-8,8,8,cv10
7,mimiciv_p8_m8_cv5,+8_-8,8,8,cv5_80_20
8,mimiciv_p3_m9_cv10,+3_-9,9,3,cv10
9,mimiciv_p3_m9_cv5,+3_-9,9,3,cv5_80_20


## Run All Grid Configurations Sequentially

This cell runs all 12 configurations one by one. The cumulative in-memory table is `grid_short_results`, where each row is one configuration and the `3h` and `6h` metrics are shown side by side.

In [22]:
import contextlib
import io

RUN_GRID = True
grid_short_results = []

if RUN_GRID:
    for idx, cfg in enumerate(EXPERIMENT_GRID, start=1):
        print(f"[{idx}/{len(EXPERIMENT_GRID)}] {cfg['task_name']}")
        run_cfg = {k: v for k, v in cfg.items() if k != 'window_label'}
        _buffer = io.StringIO()
        with contextlib.redirect_stdout(_buffer):
            result = run_experiment(**run_cfg)
        result_df = pd.DataFrame(result["results"]).sort_values("h")
        horizon_lookup = result_df.set_index("h")
        grid_short_results.append(
            {
                "task_prefix": cfg["task_prefix"],
                "window_label": cfg["window_label"],
                "split_mode": cfg["split_mode"],
                "auc_3h": horizon_lookup.loc[3, "auc_roc"] if 3 in horizon_lookup.index else pd.NA,
                "auc_6h": horizon_lookup.loc[6, "auc_roc"] if 6 in horizon_lookup.index else pd.NA,
                "auprc_3h": horizon_lookup.loc[3, "auprc"] if 3 in horizon_lookup.index else pd.NA,
                "auprc_6h": horizon_lookup.loc[6, "auprc"] if 6 in horizon_lookup.index else pd.NA,
                "total_mean_train_sec": result_df["mean_train_sec"].sum(),
                "metrics_path": result["metrics_path"],
            }
        )
        latest = pd.DataFrame(grid_short_results).tail(1)
        latest_display = latest[[
            "task_prefix",
            "window_label",
            "split_mode",
            "auc_3h",
            "auc_6h",
            "auprc_3h",
            "auprc_6h",
            "total_mean_train_sec",
        ]]
        display(latest_display)
        print(
            f"    3h AUC={latest.iloc[0]['auc_3h']:.4f} | 6h AUC={latest.iloc[0]['auc_6h']:.4f} | "
            f"3h AUPRC={latest.iloc[0]['auprc_3h']:.4f} | 6h AUPRC={latest.iloc[0]['auprc_6h']:.4f}"
        )
else:
    print("Set RUN_GRID = True when you are ready to run all 12 configurations.")


[1/12] MIMIC-IV grid | +6_-6 | cv10


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
0,mimiciv_p6_m6_cv10,+6_-6,cv10,0.966564,0.957204,0.976436,0.971099,25.40767


    3h AUC=0.9666 | 6h AUC=0.9572 | 3h AUPRC=0.9764 | 6h AUPRC=0.9711
[2/12] MIMIC-IV grid | +6_-6 | cv5


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
1,mimiciv_p6_m6_cv5,+6_-6,cv5_80_20,0.966852,0.95813,0.976625,0.971739,22.634206


    3h AUC=0.9669 | 6h AUC=0.9581 | 3h AUPRC=0.9766 | 6h AUPRC=0.9717
[3/12] MIMIC-IV grid | +3_-6 | cv10


...

    3h AUC=0.9784 | 6h AUC=0.9738 | 3h AUPRC=0.9847 | 6h AUPRC=0.9821
[11/12] MIMIC-IV grid | +0_-9 | cv10


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
10,mimiciv_p0_m9_cv10,+0_-9,cv10,0.979144,0.974356,0.985101,0.982462,30.086577


    3h AUC=0.9791 | 6h AUC=0.9744 | 3h AUPRC=0.9851 | 6h AUPRC=0.9825
[12/12] MIMIC-IV grid | +0_-9 | cv5


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
11,mimiciv_p0_m9_cv5,+0_-9,cv5_80_20,0.97799,0.974169,0.984483,0.982187,27.941361


    3h AUC=0.9780 | 6h AUC=0.9742 | 3h AUPRC=0.9845 | 6h AUPRC=0.9822


## Load the Shared Summary Table

This cell loads the cumulative horizon-level summary CSV into `summary_df`. It is the longest-form result table: each row represents one configuration at one prediction horizon, so `3h` and `6h` appear as separate rows identified by `horizon_h`.

In [26]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH).sort_values(["task_prefix", "horizon_h"]).reset_index(drop=True)
    display(summary_df)
else:
    summary_df = pd.DataFrame()
    print(f"Summary file not found: {SUMMARY_PATH}")

,task_name,task_prefix,cohort_before,cohort_after,horizon_h,split_mode,include_static_in_root,negative_alignment_mode,export_explainability,n_pos,n_neg,visible_rows_per_stay,hourly_rows,oof_auc_roc,oof_auprc,mean_fold_auc_roc,mean_fold_auprc,mean_train_sec
0,MIMIC-IV grid | +0_-6 | cv10,mimiciv_p0_m6_cv10,6,0,3,cv10,False,paper,False,7641,7641,4,61128,0.968066,0.977253,0.968202,0.977339,16.339868
1,MIMIC-IV grid | +0_-6 | cv10,mimiciv_p0_m6_cv10,6,0,6,cv10,False,paper,False,7641,7641,1,15282,0.957377,0.971418,0.957594,0.971364,10.222419
2,MIMIC-IV grid | +0_-6 | cv5,mimiciv_p0_m6_cv5,6,0,3,cv5_80_20,False,paper,False,7641,7641,4,61128,0.967205,0.976927,0.967184,0.976910,13.973106
3,MIMIC-IV grid | +0_-6 | cv5,mimiciv_p0_m6_cv5,6,0,6,cv5_80_20,False,paper,False,7641,7641,1,15282,0.957441,0.971364,0.957452,0.971223,9.804393
4,MIMIC-IV grid | +0_-9 | cv10,mimiciv_p0_m9_cv10,9,0,3,cv10,False,paper,False,5613,5613,7,78582,0.979144,0.985101,0.979290,0.985181,16.361004
5,MIMIC-IV grid | +0_-9 | cv10,mimiciv_p0_m9_cv10,9,0,6,cv10,False,paper,False,5613,5613,4,44904,0.974356,0.982462,0.974461,0.982505,13.725574
6,MIMIC-IV grid | +0_-9 | cv5,mimiciv_p0_m9_cv5,9,0,3,cv5_80_20,False,paper,False,5613,5613,7,78582,0.977990,0.984483,0.978085,0.984535,15.777768
7,MIMIC-IV grid | +0_-9 | cv5,mimiciv_p0_m9_cv5,9,0,6,cv5_80_20,False,paper,False,5613,5613,4,44904,0.974169,0.982187,0.974234,0.982243,12.163593
8,MIMIC-IV grid | +3_-6 | cv10,mimiciv_p3_m6_cv10,6,3,3,cv10,False,paper,False,7591,7591,4,60728,0.967202,0.976938,0.967288,0.976981,15.224815
9,MIMIC-IV grid | +3_-6 | cv10,mimiciv_p3_m6_cv10,6,3,6,cv10,False,paper,False,7591,7591,1,15182,0.956746,0.971044,0.956886,0.970951,10.173549


## Build the Internal Configuration Table

This cell builds `compact_summary_df` as an internal one-row-per-configuration table. It is used by the ranking and final reporting cells, so it no longer needs a separate full display.

In [27]:
if summary_df.empty:
    compact_summary_df = pd.DataFrame()
    print("Summary table is empty. Run the grid first.")
else:
    config_base_df = summary_df[[
        "task_prefix",
        "task_name",
        "cohort_before",
        "cohort_after",
        "split_mode",
        "n_pos",
        "n_neg",
        "negative_alignment_mode",
        "include_static_in_root",
        "export_explainability",
    ]].drop_duplicates(subset=["task_prefix"])
    config_base_df["window_label"] = (
        "+" + config_base_df["cohort_before"].astype(int).astype(str)
        + "_-" + config_base_df["cohort_after"].astype(int).astype(str)
    )

    horizon_metrics_df = (
        summary_df[["task_prefix", "horizon_h", "oof_auc_roc", "oof_auprc", "mean_train_sec"]]
        .pivot(index="task_prefix", columns="horizon_h", values=["oof_auc_roc", "oof_auprc", "mean_train_sec"])
        .sort_index(axis=1)
    )
    horizon_metrics_df.columns = [
        f"{metric.replace('oof_', '')}_{int(h)}h" for metric, h in horizon_metrics_df.columns
    ]
    horizon_metrics_df = horizon_metrics_df.reset_index()

    compact_summary_df = config_base_df.merge(horizon_metrics_df, on="task_prefix", how="left")
    compact_summary_df["auc_roc_delta_6h_minus_3h"] = compact_summary_df["auc_roc_6h"] - compact_summary_df["auc_roc_3h"]
    compact_summary_df["auprc_delta_6h_minus_3h"] = compact_summary_df["auprc_6h"] - compact_summary_df["auprc_3h"]
    compact_summary_df["total_mean_train_sec"] = compact_summary_df[["mean_train_sec_3h", "mean_train_sec_6h"]].sum(axis=1)
    compact_summary_df = compact_summary_df.sort_values(["auc_roc_3h", "auc_roc_6h", "auprc_3h", "auprc_6h"], ascending=False).reset_index(drop=True)
    print(f"Built compact_summary_df with {len(compact_summary_df)} configurations.")

Built compact_summary_df with 12 configurations.


## Rank the Best Configurations

This ranking view keeps `3h` and `6h` separate. Each configuration receives independent rank columns for AUC and AUPRC at both horizons.

In [28]:
TOP_K = 12

if compact_summary_df.empty:
    print("Compact summary is empty. Run the grid first.")
else:
    ranking_df = compact_summary_df[[
        "task_prefix",
        "window_label",
        "split_mode",
        "n_pos",
        "n_neg",
        "auc_roc_3h",
        "auc_roc_6h",
        "auprc_3h",
        "auprc_6h",
        "auc_roc_delta_6h_minus_3h",
        "auprc_delta_6h_minus_3h",
        "total_mean_train_sec",
    ]].copy()
    ranking_df["rank_auc_3h"] = ranking_df["auc_roc_3h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auc_6h"] = ranking_df["auc_roc_6h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auprc_3h"] = ranking_df["auprc_3h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auprc_6h"] = ranking_df["auprc_6h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df = ranking_df.sort_values(["rank_auc_3h", "rank_auc_6h", "rank_auprc_3h", "rank_auprc_6h"]).head(TOP_K)
    display(ranking_df)

,task_prefix,window_label,split_mode,n_pos,n_neg,auc_roc_3h,auc_roc_6h,auprc_3h,auprc_6h,auc_roc_delta_6h_minus_3h,auprc_delta_6h_minus_3h,total_mean_train_sec,rank_auc_3h,rank_auc_6h,rank_auprc_3h,rank_auprc_6h
0,mimiciv_p0_m9_cv10,+9_-0,cv10,5613,5613,0.979144,0.974356,0.985101,0.982462,-0.004788,-0.002640,30.086577,1,1,2,1
1,mimiciv_p3_m9_cv10,+9_-3,cv10,5575,5575,0.979027,0.973734,0.985128,0.982051,-0.005293,-0.003076,34.329047,2,4,1,4
2,mimiciv_p3_m9_cv5,+9_-3,cv5_80_20,5575,5575,0.978408,0.973838,0.984704,0.982137,-0.004570,-0.002566,29.243219,3,3,3,3
3,mimiciv_p0_m9_cv5,+9_-0,cv5_80_20,5613,5613,0.977990,0.974169,0.984483,0.982187,-0.003821,-0.002297,27.941361,4,2,4,2
4,mimiciv_p8_m8_cv10,+8_-8,cv10,5980,5980,0.977869,0.972051,0.984049,0.980699,-0.005818,-0.003350,28.011572,5,6,5,6
5,mimiciv_p8_m8_cv5,+8_-8,cv5_80_20,5980,5980,0.977365,0.972550,0.983740,0.981037,-0.004815,-0.002704,25.391600,6,5,6,5
6,mimiciv_p0_m6_cv10,+6_-0,cv10,7641,7641,0.968066,0.957377,0.977253,0.971418,-0.010689,-0.005835,26.562287,7,10,8,9
7,mimiciv_p3_m6_cv5,+6_-3,cv5_80_20,7591,7591,0.967900,0.958135,0.977328,0.971685,-0.009765,-0.005644,22.870486,8,7,7,8
8,mimiciv_p0_m6_cv5,+6_-0,cv5_80_20,7641,7641,0.967205,0.957441,0.976927,0.971364,-0.009764,-0.005564,23.777499,9,9,10,10
9,mimiciv_p3_m6_cv10,+6_-3,cv10,7591,7591,0.967202,0.956746,0.976938,0.971044,-0.010456,-0.005894,25.398364,10,12,9,12
